In [8]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, random_split, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import math
import random
from tqdm.auto import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
SYNTH_TRAIN_FIELD_DIR = '/mimer/NOBACKUP/groups/caim1/dafne/datasets/synthetic/train/b'
SYNTH_TEST_FIELD_DIR  = '/mimer/NOBACKUP/groups/caim1/dafne/datasets/synthetic/test/b'
SAVE_DIR              = '/mimer/NOBACKUP/groups/caim1/dafne/checkpoints_v2'
VIS_DIR               = '/mimer/NOBACKUP/groups/caim1/dafne/vis_v2'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(VIS_DIR,  exist_ok=True)

# ── Global constants ───────────────────────────────────────────────────────────
# Per-channel normalization derived from full training set statistics
# Channel 0 (dx LR): worst abs = 30.7  → 35.0
# Channel 1 (dy AP): worst abs = 70.9  → 75.0
# Channel 2 (dz SI): worst abs = 73.2  → 80.0
DVF_NORM = np.array([35.0, 75.0, 80.0], dtype=np.float32)  # (3,)

H, W, D  = 256, 256, 128
H_MARGIN = int(0.20 * H)   # 51 — don't condition too close to edge
W_MARGIN = int(0.20 * W)   # 51

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'DVF_NORM: {DVF_NORM}')
print(f'H_MARGIN={H_MARGIN}, W_MARGIN={W_MARGIN}')

# ── Dataset ───────────────────────────────────────────────────────────────────
class SyntheticDVFDataset(Dataset):
    """
    Loads DVF volumes and extracts random coronal + sagittal slices on the fly.
    Stored slice files are ignored — slices extracted directly from DVF
    at random positions so the model learns position-awareness.

    Returns (normalised):
        dvf:    (3, H, W, D)  float32  values in approx [-1, 1]
        cor:    (3, W, D)     float32  coronal  slice at h_idx
        sag:    (3, H, D)     float32  sagittal slice at w_idx
        h_norm: scalar float32  h_idx / (H-1)  in [0, 1]
        w_norm: scalar float32  w_idx / (W-1)  in [0, 1]
    """
    def __init__(self, field_dir, h_margin=H_MARGIN, w_margin=W_MARGIN,
                 norm=DVF_NORM, seed=None):
        self.field_dir = field_dir
        self.files     = sorted(f for f in os.listdir(field_dir)
                                if f.endswith('.npy'))
        self.h_margin  = h_margin
        self.w_margin  = w_margin
        self.norm      = norm
        self.rng       = np.random.default_rng(seed)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        dvf = np.load(os.path.join(self.field_dir,
                                   self.files[idx])).astype(np.float32)
        # dvf shape: (3, H, W, D)

        # per-channel normalisation to approx [-1, 1]
        dvf = dvf / self.norm[:, None, None, None]

        C, H_dim, W_dim, D_dim = dvf.shape

        # random slice positions within margin
        h_idx = int(self.rng.integers(self.h_margin, H_dim - self.h_margin))
        w_idx = int(self.rng.integers(self.w_margin, W_dim - self.w_margin))

        cor = dvf[:, h_idx, :, :]    # (3, W, D)
        sag = dvf[:, :, w_idx, :]    # (3, H, D)

        h_norm = np.float32(h_idx / (H_dim - 1))
        w_norm = np.float32(w_idx / (W_dim - 1))

        return (
            torch.from_numpy(dvf),
            torch.from_numpy(cor),
            torch.from_numpy(sag),
            torch.tensor(h_norm),
            torch.tensor(w_norm),
        )

# ── Splits and loaders ────────────────────────────────────────────────────────
full_train_ds = SyntheticDVFDataset(SYNTH_TRAIN_FIELD_DIR, seed=None)
test_ds       = SyntheticDVFDataset(SYNTH_TEST_FIELD_DIR,  seed=42)

val_size   = int(0.15 * len(full_train_ds))
train_size = len(full_train_ds) - val_size
train_ds, val_ds = random_split(
    full_train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds,  batch_size=2, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,    batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,   batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')

# verify one sample
dvf_s, cor_s, sag_s, h_s, w_s = full_train_ds[0]
print(f'\nSample 0:')
print(f'  dvf shape:  {tuple(dvf_s.shape)}   (expect (3, 256, 256, 128))')
print(f'  cor shape:  {tuple(cor_s.shape)}   (expect (3, 256, 128))')
print(f'  sag shape:  {tuple(sag_s.shape)}   (expect (3, 256, 128))')
print(f'  h_norm:     {h_s.item():.4f}')
print(f'  w_norm:     {w_s.item():.4f}')
print(f'  dvf range:  [{dvf_s.min():.4f}, {dvf_s.max():.4f}]  (expect within [-1, 1])')

# verify cor is actually the slice of dvf at h_idx
h_idx_check = round(h_s.item() * (H - 1))
w_idx_check = round(w_s.item() * (W - 1))
assert torch.allclose(cor_s, dvf_s[:, h_idx_check, :, :]), 'Coronal slice mismatch'
assert torch.allclose(sag_s, dvf_s[:, :, w_idx_check, :]), 'Sagittal slice mismatch'
print('  Slice consistency: PASSED')

# verify one batch
dvf_b, cor_b, sag_b, h_b, w_b = next(iter(train_loader))
print(f'\nOne train batch:')
print(f'  dvf: {tuple(dvf_b.shape)}   (expect (2, 3, 256, 256, 128))')
print(f'  cor: {tuple(cor_b.shape)}   (expect (2, 3, 256, 128))')
print(f'  sag: {tuple(sag_b.shape)}   (expect (2, 3, 256, 128))')
print(f'  h:   {h_b}')
print(f'  w:   {w_b}')

Device: cuda
DVF_NORM: [35. 75. 80.]
H_MARGIN=51, W_MARGIN=51
Train=425, Val=75, Test=100

Sample 0:
  dvf shape:  (3, 256, 256, 128)   (expect (3, 256, 256, 128))
  cor shape:  (3, 256, 128)   (expect (3, 256, 128))
  sag shape:  (3, 256, 128)   (expect (3, 256, 128))
  h_norm:     0.3686
  w_norm:     0.5216
  dvf range:  [-0.5161, 0.7028]  (expect within [-1, 1])
  Slice consistency: PASSED

One train batch:
  dvf: (2, 3, 256, 256, 128)   (expect (2, 3, 256, 256, 128))
  cor: (2, 3, 256, 128)   (expect (2, 3, 256, 128))
  sag: (2, 3, 256, 128)   (expect (2, 3, 256, 128))
  h:   tensor([0.3294, 0.4235])
  w:   tensor([0.5216, 0.3608])


In [9]:
# ── Diffusion schedule ─────────────────────────────────────────────────────────
class DiffusionSchedule:
    def __init__(self, timesteps=1000, device='cpu'):
        self.timesteps = timesteps
        self.device    = device
        betas          = torch.linspace(1e-4, 0.02, timesteps).to(device)
        alphas         = 1.0 - betas
        alpha_bar      = torch.cumprod(alphas, dim=0)
        alpha_bar_prev = F.pad(alpha_bar[:-1], (1, 0), value=1.0)

        self.betas                    = betas
        self.alphas                   = alphas
        self.alpha_bar                = alpha_bar
        self.alpha_bar_prev           = alpha_bar_prev
        self.sqrt_alphas_bar          = torch.sqrt(alpha_bar)
        self.sqrt_one_minus_alphas_bar= torch.sqrt(1.0 - alpha_bar)

    def q_sample(self, x0, t, noise):
        # standard DDPM forward — noise the entire volume
        sqrt_ab  = self.sqrt_alphas_bar[t][:, None, None, None, None]
        sqrt_oab = self.sqrt_one_minus_alphas_bar[t][:, None, None, None, None]
        return sqrt_ab * x0 + sqrt_oab * noise


# ── Time embedding ─────────────────────────────────────────────────────────────
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb  = math.log(10000) / (half - 1)
        emb  = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb  = t[:, None].float() * emb[None, :]
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


# ── SliceToVolume with learned fusion ─────────────────────────────────────────
class SliceToVolume(nn.Module):
    """
    Takes coronal (B, 3, W, D) and sagittal (B, 3, H, D) slices,
    encodes each with a small 2D CNN, broadcasts into 3D volumes,
    then fuses with a learned 3D conv instead of simple addition.

    The position (h_norm, w_norm) is encoded sinusoidally and added
    to the features before fusion so the model knows WHERE the slices are.

    Output: (B, out_channels, D, H, W)
    """
    def __init__(self, out_channels=16, pos_dim=32):
        super().__init__()
        self.out_channels = out_channels
        self.pos_dim      = pos_dim

        # 2D encoders for each slice
        self.coronal_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.GroupNorm(8, 32), nn.SiLU(),
            nn.Conv2d(32, out_channels, 3, padding=1)
        )
        self.sagittal_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.GroupNorm(8, 32), nn.SiLU(),
            nn.Conv2d(32, out_channels, 3, padding=1)
        )

        # sinusoidal position encoding → learned projection
        self.pos_mlp = nn.Sequential(
            nn.Linear(pos_dim * 2, pos_dim),   # h_emb + w_emb
            nn.SiLU(),
            nn.Linear(pos_dim, out_channels * 2)  # one scale per channel per slice
        )

        # learned fusion: takes concat of coronal and sagittal volumes
        # and produces a single fused volume
        self.fusion = nn.Sequential(
            nn.Conv3d(out_channels * 2, out_channels, 1),  # pointwise
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
        )

    def _sinusoidal(self, x, dim):
        # x: (B,) scalar, returns (B, dim)
        half  = dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=x.device) / (half - 1)
        )
        args = x[:, None] * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

    def forward(self, coronal, sagittal, h_norm, w_norm):
        """
        coronal:  (B, 3, W, D)
        sagittal: (B, 3, H, D)
        h_norm:   (B,) float in [0,1] — where the coronal slice is
        w_norm:   (B,) float in [0,1] — where the sagittal slice is
        """
        B  = coronal.shape[0]
        W  = coronal.shape[2]    # 256
        H  = sagittal.shape[2]   # 256
        D  = coronal.shape[3]    # 128

        # encode slices
        cor_feat = self.coronal_encoder(coronal)    # (B, C, W, D)
        sag_feat = self.sagittal_encoder(sagittal)  # (B, C, H, D)

        # position embedding — tells the model where the slices are
        h_emb   = self._sinusoidal(h_norm, self.pos_dim)   # (B, pos_dim)
        w_emb   = self._sinusoidal(w_norm, self.pos_dim)   # (B, pos_dim)
        pos_emb = self.pos_mlp(torch.cat([h_emb, w_emb], dim=1))  # (B, C*2)
        cor_pos = pos_emb[:, :self.out_channels]    # (B, C)
        sag_pos = pos_emb[:, self.out_channels:]    # (B, C)

        # add position to features (broadcast over spatial dims)
        cor_feat = cor_feat + cor_pos[:, :, None, None]  # (B, C, W, D)
        sag_feat = sag_feat + sag_pos[:, :, None, None]  # (B, C, H, D)

        # broadcast into 3D volumes
        # coronal: (B, C, W, D) → (B, C, D, H, W)
        cor_vol = cor_feat.permute(0, 1, 3, 2).unsqueeze(3).expand(
            -1, -1, -1, H, -1).contiguous()

        # sagittal: (B, C, H, D) → (B, C, D, H, W)
        sag_vol = sag_feat.permute(0, 1, 3, 2).unsqueeze(4).expand(
            -1, -1, -1, -1, W).contiguous()

        # learned fusion instead of simple addition
        fused = self.fusion(torch.cat([cor_vol, sag_vol], dim=1))  # (B, C, D, H, W)
        return fused


# ── UNet ───────────────────────────────────────────────────────────────────────
class ResidualBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv3d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
        )

    def forward(self, x):
        return F.silu(x + self.block(x))


class UNet3D_Diffusion(nn.Module):
    def __init__(self, cond_channels=16):
        super().__init__()
        self.time_mlp = nn.Sequential(
            TimeEmbedding(64),
            nn.Linear(64, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
        )
        self.time_proj1    = nn.Linear(128, 32)
        self.time_proj2    = nn.Linear(128, 64)
        self.time_proj_mid = nn.Linear(128, 128)
        self.time_proj_d1  = nn.Linear(128, 64)
        self.time_proj_d2  = nn.Linear(128, 32)

        self.enc1 = nn.Conv3d(3 + cond_channels, 32,  3, padding=1)
        self.enc2 = nn.Conv3d(32,                 64,  3, padding=1)
        self.pool = nn.MaxPool3d(2)

        self.mid     = nn.Conv3d(64, 128, 3, padding=1)
        self.mid_res = ResidualBlock3D(128)

        self.up2  = nn.ConvTranspose3d(128, 64, 2, stride=2)
        self.dec2 = nn.Conv3d(128, 64, 3, padding=1)   # 64 from up2 + 64 skip from e2

        self.up1  = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.dec1 = nn.Conv3d(64,  32, 3, padding=1)   # 32 from up1 + 32 skip from e1

        self.out  = nn.Conv3d(32, 3, 1)

    def forward(self, x, cond, t):
        x     = x.permute(0, 1, 4, 2, 3).contiguous()   # (B,3,D,H,W)
        t_emb = self.time_mlp(t)                          # (B,128)

        xc = torch.cat([x, cond], dim=1)                  # (B,3+C,D,H,W)

        e1 = F.silu(self.enc1(xc) +
                    self.time_proj1(t_emb)[:, :, None, None, None])
        # (B, 32, D, H, W)

        e2 = F.silu(self.enc2(self.pool(e1)) +
                    self.time_proj2(t_emb)[:, :, None, None, None])
        # (B, 64, D/2, H/2, W/2)

        m = F.silu(self.mid(self.pool(e2)) +
                   self.time_proj_mid(t_emb)[:, :, None, None, None])
        m = self.mid_res(m)
        # (B, 128, D/4, H/4, W/4)

        d2 = F.silu(self.dec2(torch.cat([self.up2(m), e2], dim=1)) +
                    self.time_proj_d1(t_emb)[:, :, None, None, None])
        # up2(m): (B,64,D/2,H/2,W/2) + e2: (B,64,D/2,H/2,W/2) → cat→(B,128,...)
        # dec2 output: (B,64,D/2,H/2,W/2)

        d1 = F.silu(self.dec1(torch.cat([self.up1(d2), e1], dim=1)) +
                    self.time_proj_d2(t_emb)[:, :, None, None, None])
        # up1(d2): (B,32,D,H,W) + e1: (B,32,D,H,W) → cat→(B,64,...)
        # dec1 output: (B,32,D,H,W)

        out = self.out(d1)
        return out.permute(0, 1, 3, 4, 2).contiguous()   # (B,3,H,W,D)


class DiffusionModelManager(nn.Module):
    def __init__(self, cond_channels=16):
        super().__init__()
        self.slice_to_vol = SliceToVolume(out_channels=cond_channels)
        self.unet         = UNet3D_Diffusion(cond_channels=cond_channels)

    def forward(self, x_t, coronal_2d, sagittal_2d, h_norm, w_norm, t):
        cond_3d = self.slice_to_vol(coronal_2d, sagittal_2d, h_norm, w_norm)
        return self.unet(x_t, cond_3d, t)


# ── Loss ───────────────────────────────────────────────────────────────────────
def compute_gradient_loss(field, penalty='l2'):
    dh = torch.abs(field[:, :, 1:, :,  :] - field[:, :, :-1, :,  :])
    dw = torch.abs(field[:, :, :,  1:, :] - field[:, :, :,  :-1, :])
    dd = torch.abs(field[:, :, :,  :, 1:] - field[:, :, :,  :, :-1])
    if penalty == 'l2':
        dh, dw, dd = dh**2, dw**2, dd**2
    return (dh.mean() + dw.mean() + dd.mean()) / 3


def diffusion_loss(model, diffusion, x0, cond_coronal, cond_sagittal,
                   h_norm, w_norm, lambda_smooth=1e-4):
    B     = x0.shape[0]
    t     = torch.randint(0, diffusion.timesteps, (B,), device=x0.device)
    noise = torch.randn_like(x0)
    x_t   = diffusion.q_sample(x0, t, noise)

    noise_pred = model(x_t, cond_coronal, cond_sagittal, h_norm, w_norm, t)

    mse_loss = F.mse_loss(noise_pred, noise)

    s_ab = diffusion.sqrt_alphas_bar[t].view(B, 1, 1, 1, 1)
    s_om = diffusion.sqrt_one_minus_alphas_bar[t].view(B, 1, 1, 1, 1)
    pred_x0   = (x_t - s_om * noise_pred) / s_ab.clamp(min=1e-6)
    pred_x0   = pred_x0.clamp(-2.0, 2.0)
    grad_loss = compute_gradient_loss(pred_x0)

    total_loss = mse_loss + lambda_smooth * grad_loss
    return total_loss, mse_loss.detach(), grad_loss.detach()


def get_smooth_lambda(epoch, total_epochs,
                      initial_lambda=1e-4, final_lambda=1e-6):
    if epoch < 200:
        return initial_lambda
    decay_range = total_epochs - 200
    step        = (initial_lambda - final_lambda) / decay_range
    return max(initial_lambda - step * (epoch - 200), final_lambda)


# ── Quick verification ─────────────────────────────────────────────────────────
diffusion = DiffusionSchedule(timesteps=1000, device=DEVICE)
model     = DiffusionModelManager(cond_channels=16).to(DEVICE)
n_params  = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

dvf_b, cor_b, sag_b, h_b, w_b = next(iter(train_loader))
dvf_b = dvf_b.to(DEVICE)
cor_b = cor_b.to(DEVICE)
sag_b = sag_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

total, mse, grad = diffusion_loss(
    model, diffusion, dvf_b, cor_b, sag_b, h_b, w_b)
print(f'total={total.item():.4f}  mse={mse.item():.4f}  '
      f'grad={grad.item():.4f}  (mse expect ~1.0 at init)')

total.backward()
grad_norms = [p.grad.norm().item()
              for p in model.parameters() if p.grad is not None]
print(f'Grad norms: min={min(grad_norms):.6f}  max={max(grad_norms):.6f}')
print(f'Params with grad: {len(grad_norms)} / '
      f'{len(list(model.parameters()))}')
print('OK')

Parameters: 1,625,251
total=1.0046  mse=1.0042  grad=3.7388  (mse expect ~1.0 at init)
Grad norms: min=0.000011  max=0.108963
Params with grad: 62 / 62
OK


In [10]:
# ── Training loop (full) ───────────────────────────────────────────────────────
def run_training(model, diffusion, train_loader, val_loader,
                 optimizer, scheduler, num_epochs, save_dir, vis_dir,
                 start_epoch=1, best_val_loss=float('inf'),
                 train_loss_history=None, val_loss_history=None,
                 train_mse_history=None, val_mse_history=None,
                 train_smooth_history=None, val_smooth_history=None):

    train_loss_history   = train_loss_history   or []
    val_loss_history     = val_loss_history     or []
    train_mse_history    = train_mse_history    or []
    val_mse_history      = val_mse_history      or []
    train_smooth_history = train_smooth_history or []
    val_smooth_history   = val_smooth_history   or []

    for epoch in range(start_epoch, num_epochs + 1):
        current_lambda = get_smooth_lambda(epoch, num_epochs)

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_total = train_mse = train_smooth = 0.0

        for dvf_b, cor_b, sag_b, h_b, w_b in train_loader:
            dvf_b = dvf_b.to(DEVICE)
            cor_b = cor_b.to(DEVICE)
            sag_b = sag_b.to(DEVICE)
            h_b   = h_b.to(DEVICE)
            w_b   = w_b.to(DEVICE)

            total_loss, mse_loss, smooth_loss = diffusion_loss(
                model, diffusion, dvf_b, cor_b, sag_b, h_b, w_b,
                lambda_smooth=current_lambda)

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_total  += total_loss.item()
            train_mse    += mse_loss.item()
            train_smooth += smooth_loss.item()

        n = len(train_loader)
        train_total  /= n
        train_mse    /= n
        train_smooth /= n
        train_loss_history.append(train_total)
        train_mse_history.append(train_mse)
        train_smooth_history.append(train_smooth)

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        val_total = val_mse = val_smooth = 0.0

        with torch.no_grad():
            for dvf_b, cor_b, sag_b, h_b, w_b in val_loader:
                dvf_b = dvf_b.to(DEVICE)
                cor_b = cor_b.to(DEVICE)
                sag_b = sag_b.to(DEVICE)
                h_b   = h_b.to(DEVICE)
                w_b   = w_b.to(DEVICE)

                total_loss, mse_loss, smooth_loss = diffusion_loss(
                    model, diffusion, dvf_b, cor_b, sag_b, h_b, w_b,
                    lambda_smooth=current_lambda)

                val_total  += total_loss.item()
                val_mse    += mse_loss.item()
                val_smooth += smooth_loss.item()

        n = len(val_loader)
        val_total  /= n
        val_mse    /= n
        val_smooth /= n
        val_loss_history.append(val_total)
        val_mse_history.append(val_mse)
        val_smooth_history.append(val_smooth)

        if scheduler is not None:
            scheduler.step()

        # ── Checkpoint ────────────────────────────────────────────────────────
        is_best = val_total < best_val_loss
        if is_best:
            best_val_loss = val_total
            torch.save(model.state_dict(),
                       os.path.join(save_dir, 'best_model.pth'))

        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'best_val_loss':        best_val_loss,
            'train_loss_history':   train_loss_history,
            'val_loss_history':     val_loss_history,
            'train_mse_history':    train_mse_history,
            'val_mse_history':      val_mse_history,
            'train_smooth_history': train_smooth_history,
            'val_smooth_history':   val_smooth_history,
            'current_lambda':       current_lambda,
        }, os.path.join(save_dir, 'latest_checkpoint.pth'))

        tag = ' *** BEST ***' if is_best else ''
        print(
            f'Epoch {epoch:03d}/{num_epochs} | λ={current_lambda:.2e} | '
            f'Train [total={train_total:.5f} mse={train_mse:.5f} '
            f'smooth={train_smooth:.5f}] | '
            f'Val [total={val_total:.5f} mse={val_mse:.5f} '
            f'smooth={val_smooth:.5f}]{tag}',
            flush=True)

        # ── Loss curves every 10 epochs ───────────────────────────────────────
        if epoch % 10 == 0:
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            ep_ax = range(1, len(train_loss_history) + 1)
            axes[0].plot(ep_ax, train_loss_history, label='Train')
            axes[0].plot(ep_ax, val_loss_history,   label='Val')
            axes[0].set_title('Total Loss')
            axes[0].legend(); axes[0].grid(True)
            axes[1].plot(ep_ax, train_mse_history, label='Train')
            axes[1].plot(ep_ax, val_mse_history,   label='Val')
            axes[1].set_title('MSE Loss')
            axes[1].legend(); axes[1].grid(True)
            axes[2].plot(ep_ax, train_smooth_history, label='Train')
            axes[2].plot(ep_ax, val_smooth_history,   label='Val')
            axes[2].set_title('Smoothness Loss')
            axes[2].legend(); axes[2].grid(True)
            plt.suptitle(f'Training curves — epoch {epoch}', fontsize=13)
            plt.tight_layout()
            plt.savefig(os.path.join(vis_dir, 'loss_curves.png'),
                        dpi=120, bbox_inches='tight')
            plt.close()

    print('Training complete.')
    return best_val_loss

In [11]:
# ── Sampler ────────────────────────────────────────────────────────────────────
@torch.no_grad()
def sample_ddpm(model, diffusion, cond_coronal, cond_sagittal,
                h_norm, w_norm, shape, device, n_steps=1000):
    """
    Standard DDPM reverse process.
    The conditioning is provided explicitly via cond_coronal and cond_sagittal
    which are passed to SliceToVolume at every step.
    No masking, no repaint — the model handles conditioning through
    the SliceToVolume branch.

    cond_coronal:  (B, 3, W, D) — normalised coronal slice
    cond_sagittal: (B, 3, H, D) — normalised sagittal slice
    h_norm:        (B,) float   — coronal position in [0,1]
    w_norm:        (B,) float   — sagittal position in [0,1]
    shape:         (B, 3, H, W, D) — desired output shape
    n_steps:       number of denoising steps (1000 = full, less = faster)
    """
    B = shape[0]

    # start from pure noise
    x = torch.randn(shape, device=device)

    # build timestep sequence
    all_steps = list(reversed(range(diffusion.timesteps)))
    if n_steps < diffusion.timesteps:
        # uniform subsampling — only valid approximation, use 1000 for best quality
        indices   = torch.linspace(0, diffusion.timesteps - 1,
                                   n_steps).long().tolist()
        all_steps = sorted(set(indices), reverse=True)

    for t_idx in tqdm(all_steps, desc='Sampling', leave=False):
        tt       = torch.full((B,), t_idx, device=device, dtype=torch.long)
        beta     = diffusion.betas[t_idx]
        alpha    = diffusion.alphas[t_idx]
        abar     = diffusion.alpha_bar[t_idx]
        abar_prev= diffusion.alpha_bar_prev[t_idx]

        # predict noise
        eps_pred = model(x, cond_coronal, cond_sagittal, h_norm, w_norm, tt)

        # predict x0
        x0_hat = (x - (1.0 - abar).sqrt() * eps_pred) / abar.sqrt().clamp(min=1e-6)
        x0_hat = x0_hat.clamp(-1.0, 1.0)

        # DDPM posterior mean
        coef1 = (abar_prev.sqrt() * beta)              / (1.0 - abar)
        coef2 = (alpha.sqrt()     * (1.0 - abar_prev)) / (1.0 - abar)
        mean  = coef1 * x0_hat + coef2 * x

        if t_idx > 0:
            var = beta * (1.0 - abar_prev) / (1.0 - abar)
            x   = mean + var.sqrt() * torch.randn_like(x)
        else:
            x   = mean

    return x   # (B, 3, H, W, D) normalised


# ── Visualisation ──────────────────────────────────────────────────────────────
def visualise_sample(dvf_gt, dvf_pred, h_norm, w_norm,
                     epoch, save_dir, tag=''):
    """
    dvf_gt, dvf_pred: (1, 3, H, W, D) normalised tensors
    Shows GT vs Pred for all 3 components in 3 views:
      - axial at D//2
      - coronal at h_idx  (conditioning plane)
      - sagittal at w_idx (conditioning plane)
    """
    gt   = dvf_gt[0].cpu().numpy()   * DVF_NORM[:, None, None, None]
    pred = dvf_pred[0].cpu().numpy() * DVF_NORM[:, None, None, None]

    h_idx = int(round(h_norm[0].item() * (H - 1)))
    w_idx = int(round(w_norm[0].item() * (W - 1)))
    d_mid = D // 2

    comp_names = ['dx', 'dy', 'dz']
    view_labels = [
        f'Axial D={d_mid}',
        f'Coronal H={h_idx} (conditioned)',
        f'Sagittal W={w_idx} (conditioned)',
    ]

    def get_views(vol):
        return [
            vol[:, :, :, d_mid],     # (3, H, W)
            vol[:, h_idx, :, :],     # (3, W, D)
            vol[:, :, w_idx, :],     # (3, H, D)
        ]

    gt_views   = get_views(gt)
    pred_views = get_views(pred)

    fig, axes = plt.subplots(6, 3, figsize=(12, 24))

    for v in range(3):
        for c in range(3):
            row_gt   = v * 2
            row_pred = v * 2 + 1

            sl_gt   = gt_views[v][c]
            sl_pred = pred_views[v][c]
            vmax    = max(np.percentile(np.abs(sl_gt), 99), 1.0)

            axes[row_gt, c].imshow(
                sl_gt.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
            axes[row_gt, c].set_title(
                f'GT {comp_names[c]} — {view_labels[v]}', fontsize=7)
            axes[row_gt, c].axis('off')

            axes[row_pred, c].imshow(
                sl_pred.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
            axes[row_pred, c].set_title(
                f'Pred {comp_names[c]} — {view_labels[v]}', fontsize=7)
            axes[row_pred, c].axis('off')

    plt.suptitle(f'Epoch {epoch}{tag} | h={h_idx} w={w_idx}', fontsize=10)
    plt.tight_layout()
    fname = os.path.join(save_dir, f'sample_epoch_{epoch:04d}{tag}.png')
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    plt.close()
    print(f'Saved: {fname}')
    return fname


# ── Quick sampler verification (untrained model = noise, that is expected) ─────
model_test   = DiffusionModelManager(cond_channels=16).to(DEVICE)
diffusion_t  = DiffusionSchedule(timesteps=1000, device=DEVICE)

dvf_b, cor_b, sag_b, h_b, w_b = next(iter(val_loader))
dvf_b = dvf_b.to(DEVICE)
cor_b = cor_b.to(DEVICE)
sag_b = sag_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

pred = sample_ddpm(model_test, diffusion_t,
                   cor_b, sag_b, h_b, w_b,
                   shape=dvf_b.shape, device=DEVICE,
                   n_steps=50)

print(f'Output shape: {tuple(pred.shape)}   (expect (1, 3, 256, 256, 128))')
print(f'Output range: [{pred.min():.4f}, {pred.max():.4f}]')
print('Sampler OK — output will look like noise until model is trained')

Sampling:   0%|          | 0/50 [00:00<?, ?it/s]

Output shape: (1, 3, 256, 256, 128)   (expect (1, 3, 256, 256, 128))
Output range: [-0.9998, 0.9998]
Sampler OK — output will look like noise until model is trained


In [12]:
# replace the overfit dataset with one that always returns center slice
class FixedSliceDVFDataset(Dataset):
    def __init__(self, field_dir, norm=DVF_NORM):
        self.field_dir = field_dir
        self.files     = sorted(f for f in os.listdir(field_dir)
                                if f.endswith('.npy'))
        self.norm      = norm

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        dvf = np.load(os.path.join(self.field_dir,
                      self.files[idx])).astype(np.float32)
        dvf = dvf / self.norm[:, None, None, None]
        C, H_dim, W_dim, D_dim = dvf.shape

        # always center
        h_idx = H_dim // 2
        w_idx = W_dim // 2

        cor = dvf[:, h_idx, :, :]
        sag = dvf[:, :, w_idx, :]

        h_norm = np.float32(h_idx / (H_dim - 1))
        w_norm = np.float32(w_idx / (W_dim - 1))

        return (
            torch.from_numpy(dvf),
            torch.from_numpy(cor),
            torch.from_numpy(sag),
            torch.tensor(h_norm),
            torch.tensor(w_norm),
        )

fixed_overfit_ds = FixedSliceDVFDataset(SYNTH_TRAIN_FIELD_DIR)
overfit_loader   = DataLoader(
    torch.utils.data.Subset(fixed_overfit_ds, list(range(5))),
    batch_size=1, shuffle=False, num_workers=0
)

In [13]:
# ── Overfit test: 5 samples, fixed center slice ────────────────────────────────
import gc
gc.collect()
torch.cuda.empty_cache()

N_OVERFIT = 5
N_EPOCHS  = 200

# Fixed slice dataset — always returns center slice for the same sample
# This is critical for the overfit test: same DVF, same conditioning, every epoch
class FixedSliceDVFDataset(Dataset):
    def __init__(self, field_dir, norm=DVF_NORM):
        self.field_dir = field_dir
        self.files     = sorted(f for f in os.listdir(field_dir)
                                if f.endswith('.npy'))
        self.norm      = norm

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        dvf = np.load(os.path.join(self.field_dir,
                      self.files[idx])).astype(np.float32)
        dvf = dvf / self.norm[:, None, None, None]
        C, H_dim, W_dim, D_dim = dvf.shape

        # always center — never random
        h_idx = H_dim // 2   # 128
        w_idx = W_dim // 2   # 128

        cor    = dvf[:, h_idx, :, :]
        sag    = dvf[:, :, w_idx, :]
        h_norm = np.float32(h_idx / (H_dim - 1))
        w_norm = np.float32(w_idx / (W_dim - 1))

        return (
            torch.from_numpy(dvf),
            torch.from_numpy(cor),
            torch.from_numpy(sag),
            torch.tensor(h_norm),
            torch.tensor(w_norm),
        )

fixed_overfit_ds = FixedSliceDVFDataset(SYNTH_TRAIN_FIELD_DIR)
overfit_loader   = DataLoader(
    torch.utils.data.Subset(fixed_overfit_ds, list(range(N_OVERFIT))),
    batch_size=1, shuffle=False, num_workers=0
)

model_ov     = DiffusionModelManager(cond_channels=16).to(DEVICE)
diffusion_ov = DiffusionSchedule(timesteps=1000, device=DEVICE)
optimizer_ov = torch.optim.AdamW(model_ov.parameters(),
                                  lr=1e-4, weight_decay=0.0)

print(f'Overfitting {N_OVERFIT} samples for {N_EPOCHS} epochs')
print(f'Conditioning: always center slice (h=128, w=128)')
print(f'{"Epoch":>6} | {"total":>8} | {"mse":>8} | {"smooth":>8}')
print('-' * 40)

for epoch in range(1, N_EPOCHS + 1):
    model_ov.train()
    e_total = e_mse = e_smooth = 0.0

    for dvf_b, cor_b, sag_b, h_b, w_b in overfit_loader:
        dvf_b = dvf_b.to(DEVICE)
        cor_b = cor_b.to(DEVICE)
        sag_b = sag_b.to(DEVICE)
        h_b   = h_b.to(DEVICE)
        w_b   = w_b.to(DEVICE)

        total, mse, smooth = diffusion_loss(
            model_ov, diffusion_ov, dvf_b, cor_b, sag_b, h_b, w_b,
            lambda_smooth=1e-4)

        optimizer_ov.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(model_ov.parameters(), 1.0)
        optimizer_ov.step()

        e_total  += total.item()
        e_mse    += mse.item()
        e_smooth += smooth.item()

    n = len(overfit_loader)
    if epoch % 20 == 0:
        print(f'{epoch:>6} | {e_total/n:>8.5f} | '
              f'{e_mse/n:>8.5f} | {e_smooth/n:>8.5f}')

# ── Sample and visualise ───────────────────────────────────────────────────────
print('\nSampling...')
model_ov.eval()

for i, (dvf_b, cor_b, sag_b, h_b, w_b) in enumerate(overfit_loader):
    dvf_b = dvf_b.to(DEVICE)
    cor_b = cor_b.to(DEVICE)
    sag_b = sag_b.to(DEVICE)
    h_b   = h_b.to(DEVICE)
    w_b   = w_b.to(DEVICE)

    with torch.no_grad():
        pred = sample_ddpm(model_ov, diffusion_ov,
                           cor_b, sag_b, h_b, w_b,
                           shape=dvf_b.shape, device=DEVICE,
                           n_steps=1000)

    visualise_sample(dvf_b, pred, h_b, w_b,
                     epoch=N_EPOCHS, save_dir=VIS_DIR,
                     tag=f'_overfit_sample{i}')

print('Done.')

Overfitting 5 samples for 200 epochs
Conditioning: always center slice (h=128, w=128)
 Epoch |    total |      mse |   smooth
----------------------------------------
    20 |  0.51902 |  0.51841 |  6.05985
    40 |  0.65187 |  0.65173 |  1.34249
    60 |  0.22017 |  0.21978 |  3.89819
    80 |  0.16006 |  0.16000 |  0.65093
   100 |  0.23091 |  0.23084 |  0.68190
   120 |  0.15162 |  0.15144 |  1.79093
   140 |  0.25448 |  0.25432 |  1.61288
   160 |  0.05969 |  0.05953 |  1.56438
   180 |  0.01831 |  0.01800 |  3.07655
   200 |  0.29962 |  0.29961 |  0.10615

Sampling...


Sampling:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: /mimer/NOBACKUP/groups/caim1/dafne/vis_v2/sample_epoch_0200_overfit_sample0.png


Sampling:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 